# NeuroLIF-SKY130 — Nominal Neuron Validation

This notebook presents the **final validated nominal TT behavior**
of the NeuroLIF-SKY130 transistor-level leaky integrate-and-fire
neuron.

Final nominal conditions:

- SKY130 1.8 V devices
- process corner: TT
- supply: 1.80 V
- temperature: 27 °C
- external membrane capacitance: 5 pF
- corrected unit-cell weighted synapse
- corrected CMOS Schmitt threshold detector
- direct spike-controlled reset
- transient command: `tran 10n 0.0004 0 1n`

This notebook reads the frozen public result artifact rather than
the earlier development/debug notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root():
    start = Path.cwd().resolve()

    candidates = [
        start,
        *start.parents,
    ]

    for candidate in candidates:

        if (
            (candidate / "results").exists()
            and
            (candidate / "layout").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate NeuroLIF-SKY130 repository root."
    )


ROOT = find_repo_root()

print("Repository root:", ROOT)


In [ ]:
import json


RESULT_FILE = (
    ROOT
    / "results"
    / "nominal"
    / "nominal_summary.json"
)


with RESULT_FILE.open(
    "r",
    encoding="utf-8",
) as f:

    nominal = json.load(f)


print(
    "POST_PATCH_NOMINAL_PASS =",
    nominal["POST_PATCH_NOMINAL_PASS"],
)


assert (
    nominal["POST_PATCH_NOMINAL_PASS"]
    is True
)


## Schmitt threshold behavior

The corrected six-transistor Schmitt detector provides a positive
hysteresis window around the membrane switching point.


In [ ]:
schmitt = nominal["schmitt"]

print(
    f"VTH_UP   = {schmitt['vth_up_V']:.6f} V"
)

print(
    f"VTH_DOWN = {schmitt['vth_down_V']:.6f} V"
)

print(
    f"Hysteresis = "
    f"{schmitt['hysteresis_mV']:.3f} mV"
)


## 2-bit weight response

The final weighted synapse uses the encoding:

| Code | Active unit current cells |
|---|---:|
| 00 | 0 |
| 01 | 1 |
| 10 | 2 |
| 11 | 3 |

The expected neuron behavior is a monotonic increase in firing
activity as the number of active unit cells increases.


In [ ]:
rows = []

for weight, data in nominal["weights"].items():

    rows.append({
        "weight":
            weight,

        "spike_count":
            data["spike_count"],

        "firing_rate_Hz":
            data["firing_rate_Hz"],

        "energy_total_J":
            data["energy_total_J"],

        "energy_per_spike_J":
            data["energy_per_spike_J"],

        "vmem_min_V":
            data["vmem_min_V"],

        "vmem_max_V":
            data["vmem_max_V"],

        "spike_max_V":
            data["spike_max_V"],
    })


nominal_df = pd.DataFrame(rows)

nominal_df


In [ ]:
fig, ax = plt.subplots(
    figsize=(7, 4.5)
)

ax.bar(
    nominal_df["weight"],
    nominal_df["spike_count"],
)

ax.set_xlabel(
    "2-bit synaptic weight"
)

ax.set_ylabel(
    "Spike count"
)

ax.set_title(
    "Final nominal TT firing response"
)

ax.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()
plt.show()


In [ ]:
checks = nominal["checks"]

for name, value in checks.items():

    print(
        f"{name:35s} : {value}"
    )


assert all(
    checks.values()
)

print()
print(
    "NOMINAL_NOTEBOOK_PASS = True"
)


## Interpretation

At the final post-patch transient resolution:

- weight `00` remains silent
- weight `01` fires
- weight `10` fires faster
- weight `11` gives the strongest response
- the firing response is monotonic
- the output spike reaches a valid high level

These are **pre-silicon simulation results**. They are not
fabricated-silicon measurements.
